In [40]:
import pandas as pd

df = pd.read_excel("copy.xlsx")

In [41]:
employed = pd.read_excel("merged.xlsx")
employed.drop(columns=["company_name", "position_name", "salary_2024" ], inplace=True)

In [ ]:
unemployed = pd.read_excel("unemployed.xlsx")
unemployed

In [ ]:
merged = pd.concat([employed, unemployed], ignore_index=True)
merged

In [ ]:
def remove_existing_records(df1, df2):
    """
    Remove records from df1 that match in df2 based on F.I.SH and cert_num/Sertifikat
    
    Args:
        df1 (pd.DataFrame): Original dataframe (49000 records) with F.I.SH and cert_num
        df2 (pd.DataFrame): Matching dataframe (2500 records) with F.I.Sh and Sertifikat
    """
    # Rename df2 columns to match df1
    df2_renamed = df2.rename(columns={
        'F.I.Sh': 'F.I.SH',
        'Sertifikat': 'cert_num'
    })
    
    # Create mask for records to keep
    mask = ~df1.set_index(['F.I.SH', 'cert_num']).index.isin(
        df2_renamed.set_index(['F.I.SH', 'cert_num']).index
    )
    
    # Return filtered dataframe
    return df1[mask]


In [46]:
# Read dataframes
df1 = df
df2 = merged

# Call function
cleaned_df = remove_existing_records(df1, df2)

# Verify results
print(f"Original records in df1: {len(df1)}")
print(f"Records to remove from df2: {len(df2)}")
print(f"Records after removal: {len(cleaned_df)}")

# # Optionally save results
cleaned_df.to_excel("cleaned_data.xlsx", index=False)

Original records in df1: 49124
Records to remove from df2: 2535
Records after removal: 46686


In [ ]:
df1 = df
df2 = merged

def find_unmatched_records(df1, df2):
    # Rename df2 columns to match df1
    df2_renamed = df2.rename(columns={
        'F.I.Sh': 'F.I.SH',
        'Sertifikat': 'cert_num'
    })
    
    # Find actual matches
    matched = df1.merge(
        df2_renamed[['F.I.SH', 'cert_num']], 
        how='inner', 
        on=['F.I.SH', 'cert_num']
    )
    
    # Find unmatched records from df2
    unmatched = df2_renamed[~df2_renamed.set_index(['F.I.SH', 'cert_num']).index.isin(
        matched.set_index(['F.I.SH', 'cert_num']).index
    )]
    
    return unmatched

# Usage
unmatched_records = find_unmatched_records(df1, df2)

# Save to Excel if needed
# unmatched_records.to_excel("unmatched_records.xlsx", index=False)

print(f"Total unmatched records: {len(unmatched_records)}")
print("\nSample of unmatched records:")
unmatched_records.head()

In [45]:
unmatched_records.to_excel("unmatched_records.xlsx", index=False)